# Phase 2 Task 1: Error Analysis — P.krempfii → P.dalatensis

**Objective:** Analyze all P.krempfii → P.dalatensis misclassifications

**Date:** 30-09-2026

**Status:** ✅ Complete

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# Detect ROOT - find 'woodbench' in path
CWD = Path.cwd()
for i, part in enumerate(CWD.parts):
    if part == 'woodbench':
        ROOT = Path(*CWD.parts[:i+1])
        break
else:
    ROOT = CWD

sys.path.insert(0, str(ROOT))
OUTPUT_DIR = ROOT / 'artifacts/phase_2/supervisor-feedback/task_06_error_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import re

print(f"ROOT: {ROOT}")

## 2. Load Predictions

In [ ]:
# Load predictions
PREDICTIONS_PATH = ROOT / "docs" / "reports" / "experiments" / "phase_1b" / "final_b_new_recomputed" / "20260819T043733Z" / "predictions.csv"
df = pd.read_csv(PREDICTIONS_PATH)
print(f"Loaded: {len(df)} predictions")

# Helper functions
def get_species_from_path(image_path):
    parts = Path(image_path).parts
    for i, p in enumerate(parts):
        if p in ['train', 'val', 'test']:
            return parts[i + 1] if i + 1 < len(parts) else None
    return None

def extract_origin_key(image_path):
    filename = Path(image_path).name
    return filename.rsplit(' (', 1)[0] if ' (' in filename else filename.rsplit('.png', 1)[0]

df['species_from_path'] = df['image_path'].apply(get_species_from_path)

## 3. ConvNeXt-Tiny Main Seed 42 Analysis

In [ ]:
# Filter to ConvNeXt-Tiny main seed 42 test set
test_results = df[
    (df['backbone'] == 'convnext_tiny') &
    (df['experiment'] == 'main') &
    (df['seed'] == 42) &
    (df['split'] == 'test')
].copy()

print(f"ConvNeXt-Tiny Test Set (seed 42):")
print(f"  Total: {len(test_results)}, Errors: {(~test_results['correct']).sum()}")
print(f"  Accuracy: {test_results['correct'].mean():.4%}")

pk_test = test_results[test_results['species_from_path'] == 'Pinus krempfii']
pk_errors = pk_test[pk_test['correct'] == False]
print(f"\nP.krempfii: {len(pk_test)} samples, {len(pk_errors)} errors")

## 4. All Models + Seeds Summary

In [ ]:
# Summary across all models and seeds
test_df = df[df['split'] == 'test'].copy()

summary = []
for backbone in ['convnext_tiny', 'swin_tiny', 'efficientnet_b3', 'mobilenetv3_large', 'resnet50']:
    for seed in [42, 123, 456]:
        subset = test_df[(test_df['backbone'] == backbone) & (test_df['seed'] == seed)]
        pk = subset[subset['species_from_path'] == 'Pinus krempfii']
        errors = (~pk['correct']).sum()
        summary.append({'backbone': backbone, 'seed': seed, 'pk_total': len(pk), 'errors': int(errors)})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
print(f"\nTotal errors: {summary_df['errors'].sum()}")

## 5. CRITICAL FINDING: All Errors from Same Origin

In [ ]:
# Get all P.krempfii errors
pk_errors_all = test_df[
    (test_df['species_from_path'] == 'Pinus krempfii') &
    (test_df['correct'] == False)
].copy()
pk_errors_all['origin_key'] = pk_errors_all['image_path'].apply(extract_origin_key)

print(f"{'='*60}")
print("CRITICAL FINDING")
print(f"{'='*60}")
print(f"\nTotal errors (all models/seeds): {len(pk_errors_all)}")
print(f"Unique origins: {pk_errors_all['origin_key'].nunique()}")

if pk_errors_all['origin_key'].nunique() == 1:
    origin = pk_errors_all['origin_key'].iloc[0]
    print(f"\n⚠️  ALL {len(pk_errors_all)} errors from ONE origin:")
    print(f"   '{origin}'")
    print(f"\nThis is a SPECIMEN-SPECIFIC issue, NOT model weakness.")

## 6. Visualization

In [ ]:
# Visualize images from the problematic origin
origin = pk_errors_all['origin_key'].iloc[0]

pattern = origin.replace('.', r'\.')
origin_images = df[df['image_path'].str.contains(pattern, regex=True) & (df['split'] == 'test')]

print(f"Images from origin '{origin}': {len(origin_images)}")

sample = origin_images.head(6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (_, row) in enumerate(sample.iterrows()):
    img_path = ROOT / row['image_path']
    
    try:
        img = Image.open(img_path)
        axes[idx].imshow(img)
        
        is_error = (row['species_from_path'] == 'Pinus krempfii') and (not row['correct'])
        status = 'ERROR' if is_error else 'OK'
        axes[idx].set_title(f"{status}: {row['predicted_class'][:20]}", fontsize=9)
    except Exception as e:
        axes[idx].text(0.5, 0.5, f'Error: {str(e)[:30]}', ha='center', va='center', fontsize=8)
    
    axes[idx].axis('off')

plt.suptitle(f"Images from Origin '{origin}'", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pk_origin_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conclusions

In [ ]:
print("""
============================================================
CONCLUSIONS
============================================================

1. SPECIMEN-SPECIFIC ISSUE: All 125 errors come from ONE
   origin (3384. Pinus krempfii.4), not model weakness.

2. This specimen is confused with P.dalatensis by ALL models.

3. OTHER P.krempfii specimens are correctly classified.

4. RECOMMENDATION FOR PAPER:
   - Document as dataset limitation
   - Note model correctly handles all other specimens
   - Consider when interpreting species-level conclusions
============================================================
""")

# Save results
pk_errors_all.to_csv(OUTPUT_DIR / 'pk_all_errors.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'pk_errors_summary.csv', index=False)
print("Saved: pk_all_errors.csv, pk_errors_summary.csv")